In [1]:
# 导入组件 Importing Components
import ipywidgets.widgets as widgets
image_widget = widgets.Image(format='jpeg', width=640, height=640)  #设置摄像头显示组件  Set up the camera display component

# 导入cv相关库 Import cv related libraries
import cv2
import numpy as np
from PIL import ImageFont
from PIL import Image
from PIL import ImageDraw
# 导入依赖包 Import dependency packages
import hyperlpr3 as lpr3
import time

In [2]:
# 将BGR图像转换为JPEG格式的字节流 Convert a BGR image to a JPEG byte stream
def bgr8_to_jpeg(value, quality=75):
    return bytes(cv2.imencode('.jpg', value)[1])

# 在图像上绘制车牌框及文字 Draw the license plate frame and text on the image
def draw_plate_on_image(img, box, text, font):
    x1, y1, x2, y2 = box
    cv2.rectangle(img, (x1, y1), (x2, y2), (225, 32, 39), 2, cv2.LINE_AA)
    cv2.rectangle(img, (x1, y1 - 20), (x2, y1), (225, 32, 39), -1)
    data = Image.fromarray(img)
    draw = ImageDraw.Draw(data)
    draw.text((x1 + 1, y1 - 18), text, (255, 255, 255), font=font)
    res = np.asarray(data)
    return res


In [3]:
import cv2
import os,socket,sys,time
import spidev as SPI
import xgoscreen.LCD_2inch as LCD_2inch
from PIL import Image,ImageDraw,ImageFont
import numpy as np
import mediapipe as mp
from numpy import linalg
from xgolib import XGO
from picamera2 import Picamera2

In [4]:
g_car = XGO(port='/dev/ttyAMA0',version="xgolite")
fm=g_car.read_firmware()
if fm[0]=='M':
    print('XGO-MINI')
    g_car = XGO(port='/dev/ttyAMA0',version="xgomini")
    dog_type='M'
elif fm[0]=='L':
    print('XGO-LITE')
    dog_type='L'
elif fm[0]=='R':
    print('XGO-RIDER')
    g_car = XGO(port='/dev/ttyAMA0',version="xgorider")
    dog_type='R'

XGO-LITE


In [5]:
#清屏
mydisplay = LCD_2inch.LCD_2inch()
mydisplay.clear()
splash = Image.new("RGB", (mydisplay.height, mydisplay.width ),"black")
mydisplay.ShowImage(splash)

In [6]:
try:
    code=0
    confidence=0
    type_idx=0
    box=0
    image=0
    display(image_widget)
    # 中文字体加载 Chinese font loading
    font_ch = ImageFont.truetype("platech.ttf", 20, 0)
    # 实例化识别对象 Instantiate the recognition object
    catcher = lpr3.LicensePlateCatcher(detect_level=lpr3.DETECT_LEVEL_LOW)#DETECT_LEVEL_HIGH640*640
    
    picam2 = Picamera2()
    picam2.configure(
        picam2.create_preview_configuration(main={"format": "RGB888", "size": (320, 240)})
    )
    picam2.start()

    pTime, cTime = 0, 0
    while True:
        frame = picam2.capture_array() 
        #frame = cv2.flip(frame, 1)
        # 执行识别算法
        results = catcher(frame)
        # 计算帧率
        cTime = time.time()
        fps = 1 / (cTime - pTime)
        pTime = cTime
        text = "FPS : " + str(int(fps))
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        # 初始化图像变量 Initialize image variables
        image = frame.copy()  # 使用原始帧作为默认图像 Use original frame as default image
        for code, confidence, type_idx, box in results:
                text = f"{code} - {confidence:.2f}"
                image = draw_plate_on_image(frame, box, text, font=font_ch)
       
        if results and len(results) > 0:
            code, confidence, _, _ = results[0]
            carcher_str = f'carcher : {code}'
            confidence_str = f'confidence: {confidence:.2f}'
        
        image_widget.value = bgr8_to_jpeg(image)

        #显示在小车的lcd屏幕上
        b,g,r = cv2.split(frame)
        img = cv2.merge((r,g,b))
        imgok = Image.fromarray(img)
        mydisplay.ShowImage(imgok)
        

        # cv2.imshow('frame', frame)
        cher_list = results[0] if results and results[0] is not None else None
        if cher_list is not None:
            print(cher_list)
        # if cv2.waitKey(1) & 0xFF == ord('q'):
        #     break

except KeyboardInterrupt:
    picam2.stop()
    picam2.close()


Image(value=b'', format='jpeg', height='640', width='640')

[0:06:32.105039404] [75495]  INFO Camera camera_manager.cpp:325 libcamera v0.3.2+99-1230f78d
[0:06:32.117545054] [157976]  INFO RPI pisp.cpp:695 libpisp version v1.0.7 28196ed6edcf 29-08-2024 (16:33:32)
[0:06:32.136580541] [157976]  INFO RPI pisp.cpp:1154 Registered camera /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 to CFE device /dev/media2 and ISP device /dev/media0 using PiSP variant BCM2712_D0
[0:06:32.141153903] [75495]  INFO Camera camera.cpp:1197 configuring streams: (0) 320x240-RGB888 (1) 640x480-GBRG_PISP_COMP1
[0:06:32.141320126] [157976]  INFO RPI pisp.cpp:1450 Sensor: /base/axi/pcie@120000/rp1/i2c@88000/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected CFE format: 640x480-PC1g
/usr/lib/python3/dist-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/lib/python3/dist-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.d

['粤R888G8', 0.9895174, 0, [72, 79, 194, 127]]
['粤R888G8', 0.9960434, 0, [69, 74, 192, 118]]
['粤R888G8', 0.9976627, 0, [70, 67, 194, 111]]
['粤R888G8', 0.9960727, 0, [71, 67, 197, 110]]
['粤R888G8', 0.9926939, 0, [72, 68, 200, 113]]
['粤R888G8', 0.98943603, 0, [76, 71, 204, 115]]
['粤R888G8', 0.9963239, 0, [75, 72, 204, 116]]
['粤R888G8', 0.9943985, 0, [75, 74, 202, 118]]
['粤R888G8', 0.99455404, 0, [74, 72, 201, 116]]
['粤R888G8', 0.98558015, 0, [75, 68, 200, 111]]
['粤R888G8', 0.99594545, 0, [76, 69, 201, 113]]
['粤R888G8', 0.98365724, 0, [77, 76, 202, 120]]
['粤R888G8', 0.99208885, 0, [83, 84, 207, 125]]


In [7]:
#最后需要释放掉摄像头的占用 Finally, you need to release the camera's occupancy
picam2.stop()
picam2.close()

mydisplay.clear()
splash = Image.new("RGB", (mydisplay.height, mydisplay.width ),"black")
mydisplay.ShowImage(splash)

del g_car